# Week 36: Data and rule-based baselines

## Load the data

In [24]:
import re
from collections import Counter

import numpy as np
import nltk
import pandas as pd

from datasets import load_dataset
from sklearn.metrics import precision_recall_fscore_support

nltk.download("punkt_tab", quiet=True)

dataset = load_dataset("coastalcph/tydi_xor_rc")

df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

df_train_ar = df_train[df_train["lang"] == "ar"]
df_train_ko = df_train[df_train["lang"] == "ko"]
df_train_te = df_train[df_train["lang"] == "te"]

df_validation_ar = df_validation[df_validation["lang"] == "ar"]
df_validation_ko = df_validation[df_validation["lang"] == "ko"]
df_validation_te = df_validation[df_validation["lang"] == "te"]

print(dataset)
print("Columns:", dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['question', 'context', 'lang', 'answerable', 'answer_start', 'answer', 'answer_inlang'],
        num_rows: 15343
    })
    validation: Dataset({
        features: ['question', 'context', 'lang', 'answerable', 'answer_start', 'answer', 'answer_inlang'],
        num_rows: 3011
    })
    test: Dataset({
        features: ['question', 'context', 'lang', 'answerable', 'answer_start', 'answer', 'answer_inlang'],
        num_rows: 4
    })
})
Columns: ['question', 'context', 'lang', 'answerable', 'answer_start', 'answer', 'answer_inlang']


## 1. Dataset statistics

In [25]:
def tokenize(text):
    return nltk.word_tokenize(text)


def get_statistics(df, split, language):
    question_lengths = []
    context_lengths = []

    for question in df["question"]:
        question_lengths.append(len(tokenize(question)))

    for context in df["context"]:
        context_lengths.append(len(tokenize(context)))

    answerable_n = len(df[df["answerable"] == True])
    unanswerable_n = len(df[df["answerable"] == False])

    return {
        "split": split,
        "language": language,
        "n": len(df),
        "answerable_%": 100 * answerable_n / len(df),
        "unanswerable_%": 100 * unanswerable_n / len(df),
        "question_median": np.median(question_lengths),
        "question_IQR": np.percentile(question_lengths, 75) - np.percentile(question_lengths, 25),
        "context_median": np.median(context_lengths),
        "context_IQR": np.percentile(context_lengths, 75) - np.percentile(context_lengths, 25)
    }


statistics = pd.DataFrame([
    get_statistics(df_train_ar, "train", "ar"),
    get_statistics(df_train_ko, "train", "ko"),
    get_statistics(df_train_te, "train", "te"),
    get_statistics(df_validation_ar, "validation", "ar"),
    get_statistics(df_validation_ko, "validation", "ko"),
    get_statistics(df_validation_te, "validation", "te")
])

statistics.round(1)

,split,language,n,answerable_%,unanswerable_%,question_median,question_IQR,context_median,context_IQR
0,train,ar,2558,90.0,10.0,6.0,3.0,103.0,80.0
1,train,ko,2422,97.4,2.6,6.0,2.0,99.0,73.0
2,train,te,1355,96.7,3.3,6.0,3.0,93.0,71.0
3,validation,ar,415,87.5,12.5,6.0,3.0,103.0,77.0
4,validation,ko,356,94.7,5.3,6.0,2.0,97.5,85.2
5,validation,te,384,75.8,24.2,7.0,2.0,120.0,76.0


### Missing values and duplicates

In [ ]:
overall_rows = [] # All languages, not just ar, ko, te

for split, df in [("train", df_train), ("validation", df_validation)]:
    answerable_n = len(df[df["answerable"] == True])
    unanswerable_n = len(df[df["answerable"] == False])

    overall_rows.append({
        "split": split,
        "n": len(df),
        "answerable_%": 100 * answerable_n / len(df),
        "unanswerable_%": 100 * unanswerable_n / len(df)
    })

pd.DataFrame(overall_rows).round(1)

,split,n,answerable_%,unanswerable_%
0,train,15343,91.0,9.0
1,validation,3011,76.8,23.2


In [34]:
data_quality_rows = []

dataframes = [
    ("train", "ar", df_train_ar),
    ("train", "ko", df_train_ko),
    ("train", "te", df_train_te),
    ("validation", "ar", df_validation_ar),
    ("validation", "ko", df_validation_ko),
    ("validation", "te", df_validation_te)
]

for split, language, df in dataframes:
    missing_values = df.isna().sum().sum()
    duplicate_pairs = df[["question", "context"]].duplicated().sum()

    data_quality_rows.append({
        "split": split,
        "language": language,
        "missing_values": missing_values,
        "duplicate_question_context_pairs": duplicate_pairs
    })

pd.DataFrame(data_quality_rows)

,split,language,missing_values,duplicate_question_context_pairs
0,train,ar,2558,0
1,train,ko,2422,10
2,train,te,1305,0
3,validation,ar,415,0
4,validation,ko,356,0
5,validation,te,284,0


## 2. Most common question tokens

In [35]:
def most_common_tokens(df):
    tokens = []

    for question in df["question"]:
        tokens.extend(tokenize(question))

    return Counter(tokens).most_common(5)


ar_top5 = most_common_tokens(df_train_ar)
ko_top5 = most_common_tokens(df_train_ko)
te_top5 = most_common_tokens(df_train_te)

print("Arabic:", ar_top5)
print("Korean:", ko_top5)
print("Telugu:", te_top5)

Arabic: [('؟', 1083), ('في', 593), ('من', 587), ('متى', 536), ('ما', 443)]
Korean: [('?', 2420), ('가장', 527), ('무엇인가', 497), ('언제', 336), ('몇', 234)]
Telugu: [('?', 1355), ('ఎవరు', 274), ('ఏది', 192), ('ఎన్ని', 165), ('ఎప్పుడు', 154)]


## 3. Verify answer spans

In [37]:
span_rows = []

df_train_selected = df_train[df_train["lang"].isin(["ar", "ko", "te"])]
df_validation_selected = df_validation[df_validation["lang"].isin(["ar", "ko", "te"])]

dataframes = [
    ("train", "ar", df_train_ar),
    ("train", "ko", df_train_ko),
    ("train", "te", df_train_te),
    ("train", "all", df_train_selected),
    ("validation", "ar", df_validation_ar),
    ("validation", "ko", df_validation_ko),
    ("validation", "te", df_validation_te),
    ("validation", "all", df_validation_selected)
]

for split, language, df in dataframes:
    answerable = df[df["answerable"] == True]
    failures = 0

    for context, start, answer in zip(
        answerable["context"],
        answerable["answer_start"],
        answerable["answer"]
    ):
        start = int(start)

        if context[start:start + len(answer)] != answer:
            failures += 1

    span_rows.append({
        "split": split,
        "language": language,
        "checked": len(answerable),
        "failures": failures
    })

pd.DataFrame(span_rows)

,split,language,checked,failures
0,train,ar,2303,0
1,train,ko,2359,0
2,train,te,1310,0
3,train,all,5972,0
4,validation,ar,363,0
5,validation,ko,337,0
6,validation,te,291,0
7,validation,all,991,0
